In [11]:
import numpy as np
import pandas as pd
from scipy import sparse
import re

from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.tree import DecisionTreeRegressor

In [12]:
x_train = pd.read_csv("x_train.csv")
y_train = pd.read_csv("y_train.csv")
x_test  = pd.read_csv("x_test.csv")

In [13]:
# Preprocess for the train

x = x_train.copy()
y = y_train["virality_score"]

# 1.Fill missing for numeric, categorial, text features
train_followers_at_upload_median = x["followers_at_upload"].median()
train_video_duration_sec_median = x["video_duration_sec"].median()
x["followers_at_upload"] = x["followers_at_upload"].fillna(train_followers_at_upload_median)
x["video_duration_sec"] = x["video_duration_sec"].fillna(train_video_duration_sec_median)

# sound_name has no missing v
categorical_with_missings = ["visual_tags"]
# "unknown_visual_tags" or "unknown_user_region" instead of NA so it is now a category
for feature in categorical_with_missings:
  x[feature] = x[feature].fillna(f"unknown_{feature}")

# user_rigion: author_bio's suffix(last token) of author_bio implies the user_region
# completes missing "user_region" values with a likely region, by "author_bio"'s last token.
token_pat = re.compile(r"[^\W_]+", flags=re.UNICODE)
def last_token(s):
    if pd.isna(s):
        return ""
    tokens = token_pat.findall(s)
    if len(tokens) == 0:
        return ""
    return tokens[-1].lower()
author_bio_suffixes = { }
for bio, region in zip(x["author_bio"], x["user_region"]):
    if pd.isna(region):
      continue

    suffix = last_token(bio)
    if suffix == "":
      continue
    region_key = region #if not pd.isna(region) else "NaN"

    if suffix not in author_bio_suffixes:
        author_bio_suffixes[suffix] = {}

    author_bio_suffixes[suffix][region_key] = author_bio_suffixes[suffix].get(region_key, 0) + 1

# suffix -> [[freq, region1], [freq, region2], ...]
author_bio_suffixes_list = {
    suf: [[cnt, reg] for reg, cnt in sorted(reg_dict.items(), key=lambda x: x[1], reverse=True)]
    for suf, reg_dict in author_bio_suffixes.items()
}
# it seems that suffix that have more than 50 frequencies are indicative
filtered = {}
for k, v in author_bio_suffixes_list.items():
    if k != "" and v[0][0] > 50:
        filtered[k] = v

filtered_for_processing = {suf: v[0][1] for suf, v in filtered.items()}

for idx, row in x[["user_region", "author_bio"]].iterrows():
    if pd.isna(row["user_region"]):
        suffix = last_token(row["author_bio"])
        if suffix in filtered_for_processing:
            x.at[idx, "user_region"] = filtered_for_processing[suffix]

x["user_region"] = x["user_region"].fillna("unknown_user_region")

# For text features - empty string " " instead of NA - when create subject, empty string will represent 0 connection to the subject
x["author_bio"] = x["author_bio"].fillna("").astype(str)
x["video_transcription_text"] = x["video_transcription_text"].fillna("").astype(str)

# 2. Add the 2% feature to represent the effect of the lowest 2% of followers_at_upload on the y
# each sample who is in the lower 2% gets 1, other 0. by threshold
train_2_percent_threshold = x["followers_at_upload"].quantile(0.02)
x["followers_at_upload_low_2_percent"] = (x["followers_at_upload"] <= train_2_percent_threshold).astype(int)

# 3. Timestamp's upload hour and upload day as features themselves (showed the connection to y at the EDA)
timestamp_info = pd.to_datetime(x["creation_timestamp"], errors="coerce")
hour = timestamp_info.dt.hour
day = timestamp_info.dt.dayofweek
# The representation: for each hour/day, represent by its location on the unit circle. that can ensure the closeness between 0 to 23 for example
# (sin,cos) represents the location and shows which hours/ days are truly close
# regular numbers won't cover this
hour_sin = np.sin(2 * np.pi * hour / 24.0)
hour_cos = np.cos(2 * np.pi * hour / 24.0)
day_sin  = np.sin(2 * np.pi * day / 7.0)
day_cos  = np.cos(2 * np.pi * day / 7.0)
# Add as features
x["hour_sin"] = hour_sin
x["hour_cos"] = hour_cos
x["day_sin"]  = day_sin
x["day_cos"]  = day_cos

# 4. From uploader_username we take an indicator of how many times the user uploaded video
# saw at the EDA that there is a connection between user that uploaded more times to higher virality score
upload_counts = x["uploader_username"].value_counts()
num_uploads = x["uploader_username"].map(upload_counts)
x["num_uploads_by_user"] = num_uploads

# 4.2 : Conclude the numerical features
numeric_features = ["followers_at_upload", "video_duration_sec", "hour_sin", "hour_cos", "day_sin", "day_cos", "num_uploads_by_user", "followers_at_upload_low_2_percent"]
x_numeric = sparse.csr_matrix(x[numeric_features].to_numpy(dtype=float))


# 5. Categorical features become to one-hot
categorical_features = ["sound_name", "visual_tags", "user_region"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
x_categorical_encoded = encoder.fit_transform(x[categorical_features])

# 6. Text features: those have endless options (free text). there is no connection between text structure to y (EDA).
# We want to give importance to the content without adding thousands of features and overfitt.
# Use tf-idf and mnf to define topics, then add features as topic1...topic10 (defined)
# Each sample will get a score of connection to each topic.
# The model can take into consideration the content (the topics) of the texts
# Combine author_bio and video_transcription_text and create topics by the combination because the bio and videos topics may be related
# Can create the topics as total viral topics no matter that came from bio or transcript text.

# TF-IDF is used to represent texts as word-importance vectors (only top 3000 words),
# and NMF reduces this representation into a set of semantic topics.
# The output assigns each text a weight for every topic, indicating how strongly the text is related to that topic.

text = (x["author_bio"] + " " + x["video_transcription_text"])
tfidf = TfidfVectorizer(max_features=3000, min_df = 2, stop_words = "english")
text_tfidf = tfidf.fit_transform(text)
nmf = NMF(n_components=10)
x_by_topics_dense = nmf.fit_transform(text_tfidf)
# To connect with the other features
x_topics = sparse.csr_matrix(x_by_topics_dense)

# Final: create the train matrix : combination of all the parts
# Without: video_id, uploader_username, creation_timestamp. already took the relevant parts.
x_train_processed = sparse.hstack([x_numeric, x_categorical_encoded, x_topics], format="csr")


In [14]:
# Preprocess for x_test: to make it the same format as the model knows (format of x_train_processed)
# That means: we create the needed extra features and ignore the ones who we didn't keep it the train preprocess
# Note: the features we create at the x_test : their values will be filled by x_train values
# That because we assume we have no knowledge about the whole test data (when comes to features like "num_uploads_by_user" and "followers_at_upload_low_2_percent" )
# Same idea, the missing values filling will be identical to the train's data (take the train's median for example)

x_test_prep = x_test.copy()

#1. Fill missing values
x_test_prep["followers_at_upload"] = x_test_prep["followers_at_upload"].fillna(train_followers_at_upload_median)
x_test_prep["video_duration_sec"] = x_test_prep["video_duration_sec"].fillna(train_video_duration_sec_median)

# For categorical - "unknown_{feature}" instead of NA ,so it is now a category
x_test_prep["sound_name"] = x_test_prep["sound_name"].fillna("unknown_sound_name")
x_test_prep["visual_tags"] = x_test_prep["visual_tags"].fillna("unknown_visual_tags")

x_test_prep["author_bio"] = x_test_prep["author_bio"].fillna("").astype(str)

# for user region:
mask_missing_region = x_test_prep["user_region"].isna()
suffixes_test = x_test_prep.loc[mask_missing_region, "author_bio"].map(last_token)
inferred_regions = suffixes_test.map(filtered_for_processing)
x_test_prep.loc[mask_missing_region & inferred_regions.notna(),"user_region"] = inferred_regions
x_test_prep["user_region"] = x_test_prep["user_region"].fillna("unknown_user_region")

# sum up categories:
categorical_for_test = ["sound_name", "visual_tags", "user_region"]


# For text features - empty string " " instead of NA - when create subject, empty string will represent 0 connection to the subject
x_test_prep["author_bio"] = x_test_prep["author_bio"].fillna("").astype(str)
x_test_prep["video_transcription_text"] = x_test_prep["video_transcription_text"].fillna("").astype(str)

# 2. Add the 2% feature (by train 2% threshold calculation) to represent the effect of the lowest 2% of followers_at_upload on the y
x_test_prep["followers_at_upload_low_2_percent"] = (x_test_prep["followers_at_upload"] <= train_2_percent_threshold).astype(int)


# 3. Timestamp's upload hour and upload day as features themselves : by the test data because it's only technical seperation:
timestamp_info_test = pd.to_datetime(x_test_prep["creation_timestamp"], errors="coerce")
hour_test = timestamp_info_test.dt.hour
day_test = timestamp_info_test.dt.dayofweek
# The representation: for each hour/day, represent by its location on the unit circle. that can ensure the closeness between 0 to 23 for example
# (sin,cos) represents the location and shows which hours/ days are truly close
# regular numbers won't cover this
hour_sin = np.sin(2 * np.pi * hour_test / 24.0)
hour_cos = np.cos(2 * np.pi * hour_test / 24.0)
day_sin  = np.sin(2 * np.pi * day_test / 7.0)
day_cos  = np.cos(2 * np.pi * day_test / 7.0)
# add as features
x_test_prep["hour_sin"] = hour_sin
x_test_prep["hour_cos"] = hour_cos
x_test_prep["day_sin"]  = day_sin
x_test_prep["day_cos"]  = day_cos

# 4. From uploader_username we take an indicator of how many times the user uploaded video
# Note: that is calculated by train data only! we fill by the username's info from the train data
# If user didn't appear in the train, we fill with value of 1 (general non-noisy for this feature)
# "upload_counts" calculated in train
num_uploads_test = x_test_prep["uploader_username"].map(upload_counts).fillna(1)
x_test_prep["num_uploads_by_user"] = num_uploads_test

# 4.2 : Conclude the numerical features
numeric_features = ["followers_at_upload", "video_duration_sec", "hour_sin", "hour_cos", "day_sin", "day_cos", "num_uploads_by_user", "followers_at_upload_low_2_percent"]
x_numeric_test = sparse.csr_matrix(x_test_prep[numeric_features].to_numpy(dtype=float))

# 5. Categorical features become to one-hot: same encoder as train to have same features

x_categorical_encoded_test = encoder.transform(x_test_prep[categorical_for_test])

#6. Text - dame as train preprocess: tfidf + mnf: use of the same tfidf and mnf objects as train to get the sampe topics
test_text = (x_test_prep["author_bio"] + " " + x_test_prep["video_transcription_text"])
x_test_tfidf = tfidf.transform(test_text)
x_test_by_topics_dense = nmf.transform(x_test_tfidf)
x_topics_test = sparse.csr_matrix(x_test_by_topics_dense)

# Final:
x_test_processed = sparse.hstack([x_numeric_test, x_categorical_encoded_test, x_topics_test], format="csr")










In [15]:
# Train the model by train data only, according to the optimal hyper parameters we got in the train section at the original notebook.
# We just create the regression tree by those parameters and train the model on all the train data.
# Best hyperparameters: {'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_depth': 20, 'ccp_alpha': 1e-05}
final_tree = DecisionTreeRegressor(min_samples_split=2, min_samples_leaf=10, max_features=0.5, max_depth=20, ccp_alpha=1e-05)
final_tree.fit(x_train_processed, y)



DecisionTreeRegressor(ccp_alpha=1e-05, max_depth=20, max_features=0.5,
                      min_samples_leaf=10)

In [16]:
# Predict the results for x_test
test_predicts = final_tree.predict(x_test_processed)

In [17]:
# Submission creation csv
submission = pd.DataFrame({"video_id": x_test["video_id"], "virality_score": test_predicts})
submission.to_csv("submission.csv", index=False)